In [2]:
import json
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

project_root = Path.cwd().parent
os.chdir(project_root)

from src.rag_pipeline import RagPipeline
from src.generator import QwenGenerator

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
rag = RagPipeline()
generator = QwenGenerator()
with open("data/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [5]:
import random


random.seed(42)
sample_size = min(20, len(qa_data))  # Evaluate on 20 samples
eval_samples = random.sample(qa_data, sample_size)

In [6]:
results = []

for sample in tqdm(eval_samples, desc="Generating climate answers"):
    question = sample["question"]
    reference = sample["answer"]

    # Retrieve relevant documents using RAG
    retrieved = rag.retrieve(question, top_k=3)
    context_rag = "\n".join(
        f"[{i}] Q: {r['question']}\nA: {r['answer']}"
        for i, r in enumerate(retrieved, 1)
    )[:2500]

    # Generate with RAG
    answer_rag = generator.generate(question, context=context_rag, max_tokens=128)

    # Generate without RAG
    answer_baseline = generator.generate(question, context="", max_tokens=128)

    results.append(
        {
            "question": question,
            "reference": reference,
            "answer_rag": answer_rag,
            "answer_baseline": answer_baseline,
            "sources": [r["question"] for r in retrieved],
            "source_scores": [r["score"] for r in retrieved],
        }
    )

Generating climate answers: 100%|██████████| 20/20 [02:07<00:00,  6.37s/it]


In [7]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Initialize ROUGE scorer
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def compute_metrics(reference, generated):
    """Compute ROUGE and BLEU metrics for climate Q&A evaluation."""
    # ROUGE scores
    rouge_scores = rouge.score(reference, generated)

    # BLEU score
    ref_tokens = reference.split()
    gen_tokens = generated.split()
    smoothie = SmoothingFunction().method4
    bleu = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoothie)

    return {
        "rouge1": rouge_scores["rouge1"].fmeasure,
        "rouge2": rouge_scores["rouge2"].fmeasure,
        "rougeL": rouge_scores["rougeL"].fmeasure,
        "bleu": bleu,
    }

In [9]:
metrics_rag = []
metrics_baseline = []

for result in tqdm(results, desc="Computing climate metrics"):
    metrics_rag.append(compute_metrics(result["reference"], result["answer_rag"]))
    metrics_baseline.append(
        compute_metrics(result["reference"], result["answer_baseline"])
    )

Computing climate metrics: 100%|██████████| 20/20 [00:00<00:00, 333.27it/s]


## Step 5: Analyze Results

In [13]:
import numpy as np


def average_metrics(metrics_list):
    """Compute average metrics across evaluation samples."""
    return {
        metric: np.mean([m[metric] for m in metrics_list])
        for metric in metrics_list[0].keys()
    }


# Calculate averages
avg_rag = average_metrics(metrics_rag)
avg_baseline = average_metrics(metrics_baseline)

comparison = pd.DataFrame({"RAG": avg_rag, "Baseline": avg_baseline}).T

comparison.round(4)

,rouge1,rouge2,rougeL,bleu
RAG,0.5722,0.4982,0.5372,0.3314
Baseline,0.2426,0.0964,0.1764,0.0416
